# 🎬 Montana State University M090: Recordly Motion Engine (Google Colab GPU Edition)

교수들과 학생들이 극찬한 **카메라 스프링 줌·패닝, 좌·우 패널 포커스, 턴 트래킹 네온 링, 스튜디오 윈도우 프레임**을 그대로 탑재한 **공식 Recordly 비디오 연출 엔진**을 **Google 무료 T4 GPU** 환경에서 초고속으로 구동하는 원클릭 노트북입니다.

---
### 🌟 핵심 연출 아키텍처
1. **좌측 패널 포커스 (`Scale 1.42, PanX +22%`)**: 문제 정의, 칠판 수식, 정의 설명에 시선 집중
2. **우측 패널 포커스 (`Scale 1.42, PanX -22%`)**: 단계별 AI 풀이 및 그래프 계산 과정 확대
3. **조교 Sora의 함정/꿀팁 포커스 (`Scale 1.42, PanY -22%`)**: 필수 주의사항 및 황금률 카드 강조
4. **글로벌 오버뷰 (`Scale 1.0, Pan 0, 0`)**: 슬라이드 도입부 및 단원 마무리 풀샷
5. **Slide 1 세로 추적**: 오리엔테이션 대화 턴(Turn 0~5)에 맞춘 실시간 네온 링 강조 & Y축 패닝
6. **Google 무료 T4 GPU 가속 (`h264_nvenc`)**: 고화질 1080p 60fps 비디오 초고속 인코딩

## 1단계: 무료 GPU 활성화 및 Linux 가상 디스플레이(Xvfb) 환경 구축
상단 메뉴의 **[런타임] → [런타임 유형 변경]**에서 하드웨어 가속기가 **T4 GPU**로 지정되어 있는지 확인하세요.

In [ ]:
# 1. GPU 하드웨어 점검
!nvidia-smi

# 2. 리눅스 가상 디스플레이(Xvfb) 및 브라우저 의존성 설치
!apt-get update -qq && apt-get install -y -qq xvfb libxkbcommon-x11-0 libgbm-dev

# 3. 파이썬 라이브러리 및 Playwright Chromium 설치
!pip install -q edge-tts imageio-ffmpeg playwright
!playwright install chromium
!playwright install-deps

## 2단계: 가상 디스플레이(Xvfb) 실행 & 저장소 클론

In [ ]:
import os
import subprocess

# 1920x1080 60fps 가상 디스플레이 가동
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1920x1080x24"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ["DISPLAY"] = ":99"
print("🖥️ Linux 가상 디스플레이 (:99, 1920x1080) 시작 완료!")

# GitHub 프로젝트 클론
%cd /content
if not os.path.exists('/content/montana-lecture-app'):
    !git clone https://github.com/marylatte11-svg/montana-lecture-app.git /content/montana-lecture-app
%cd /content/montana-lecture-app
!pwd

## 3단계 (선택): 구글 드라이브 마운트
완성된 강의 마스터 영상을 구글 드라이브에 자동 보관하려면 실행하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/MSU_Recordly_Lectures'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"☁️ 구글 드라이브 저장 경로: {DRIVE_DIR}")

## 4단계: Recordly 카메라 패닝 엔진 실행 (원클릭 렌더링)
`--lecture` 인자에 원하는 강의 번호(1~45)를 지정하고 `--all` 옵션을 주면 전체 슬라이드를 차례로 렌더링한 후 풀 마스터 비디오로 자동 결합합니다.

In [ ]:
LECTURE_NUM = 1

# T4 GPU NVENC 하드웨어 가속으로 Recordly 카메라 모션 렌더링 실행
!python scripts/recordly_engine/render_montana_recordly.py --lecture {LECTURE_NUM} --all

## 5단계: 완성된 마스터 비디오 다운로드 및 구글 드라이브 백업

In [ ]:
import glob
import shutil
from google.colab import files

master_files = glob.glob(f"/content/montana-lecture-app/Montana_State_Univ/recordly_videos/Lecture{LECTURE_NUM:02d}/*_Full_Master.mp4")

if master_files:
    target_video = master_files[0]
    size_mb = os.path.getsize(target_video) / (1024 * 1024)
    print(f"🎬 완성된 Recordly 마스터 비디오: {os.path.basename(target_video)} ({size_mb:.2f} MB)")
    
    # 구글 드라이브 백업
    if os.path.exists('/content/drive/MyDrive'):
        dest = os.path.join(DRIVE_DIR, os.path.basename(target_video))
        shutil.copy2(target_video, dest)
        print(f"☁️ 구글 드라이브 업로드 완료: {dest}")
        
    # 로컬 다운로드
    print("⬇️ 브라우저 다운로드를 시작합니다...")
    files.download(target_video)
else:
    print("❌ 비디오 파일을 찾을 수 없습니다. 4단계를 먼저 실행해 주세요.")